In [27]:
# Import necessary libraries
import os
import sys
import shutil
import glob

import cv2
import numpy as np
import yaml
from tqdm.contrib.concurrent import thread_map# or thread_map

In [28]:
def create_project_file(file_pth: str, dataset_pth: str, label_mapping: dict) -> None:
    data = {
        "path": dataset_pth,
        "train": "images/train",
        "val": "images/val",
        "kpt_shape": [1, 2],
    }

    with open(file_pth, "wt") as f:
        yaml.dump(data, f, sort_keys=False, default_flow_style=None)
    
    with open(file_pth, "at") as f:
        yaml.dump({"names": label_mapping}, f, sort_keys=False)

In [29]:
def get_files_glob(directory, ext="*.txt"):
    pattern = os.path.join(directory, '**', ext)
    txt_files = glob.glob(pattern, recursive=True)
    return txt_files

In [30]:
BASE_PTH = "/home/kyamkovyi/projects/researches/people_detection_pg2/quality_reduce_comparison"
BASE_DATASET = "cam4_poc_part1_15"
FILTERED_DATASET = "cam4_poc_part1_15_blurred_grayscale"

SUBSETS = ["train", "val"]

In [31]:
src_pth = os.path.join(BASE_PTH, BASE_DATASET)
dst_pth = os.path.join(BASE_PTH, FILTERED_DATASET)
if os.path.exists(dst_pth):
    shutil.rmtree(dst_pth)
shutil.copytree(src_pth, dst_pth)

cache_files = get_files_glob(dst_pth, "*.cache")
for i in cache_files:
    os.remove(i)

In [32]:
# create project file
create_project_file(os.path.join(dst_pth, "project.yaml"), dst_pth, {0: "person"})

In [33]:
img_pths = get_files_glob(os.path.join(dst_pth, "images"), "*.jpg")
len(img_pths)

152

In [34]:
def process_file(pth):
    img = cv2.imread(pth)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    kernel_size = (3, 3)
    gray = cv2.GaussianBlur(gray, kernel_size, 0)
    cv2.imwrite(pth, gray)

    return 1

In [35]:
r = thread_map(process_file, img_pths, max_workers=8)

count = 0
for i in r:
    count += i

print(count == len(img_pths))

  0%|          | 0/152 [00:00<?, ?it/s]

100%|██████████| 152/152 [00:00<00:00, 2320.69it/s]

True
